# Trabajo Práctico Aprendizaje Automático 2 - Predicción de puntuación y capitalización en texto normalizado

In [1]:
import re
from datasets import load_dataset
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split   
import unicodedata


In [2]:
# Seteamos la semilla para que los resultados sean reproducibles
SEED = 2026
np.random.seed(SEED)

In [3]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(119547, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p

## Datasets

Parlamento europeo dataset (parte en español): https://statmt.org/europarl/

Combinamos dos fuentes de datos en español para tener diversidad lingüística:

1. **Parlamento Europeo** (`Helsinki-NLP/europarl`): Texto formal, actas parlamentarias (parte en español). Fuente: [Europarl](https://statmt.org/europarl/)
2. **OpenSubtitles** (vía HuggingFace `Helsinki-NLP/open_subtitles`): Subtítulos de películas/series, lenguaje conversacional e informal (parte en español). Fuente: [Conversational Datasets (PolyAI)](https://github.com/PolyAI-LDN/conversational-datasets)

In [4]:
# Parámetros del dataset
N_TOTAL = 15000          # Total de oraciones objetivo
MIN_WORDS = 8             # Mínimo de palabras por oración
MAX_WORDS = 40            # Máximo de palabras por oración

In [35]:
SIGNOS_PROHIBIDOS = set("!¡;:\"`«»—–…[]{}()")

def normalizar_espacios_puntuacion(texto):
    # Elimina espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    # Quita espacios después de '¿' (ej: "¿ texto" -> "¿texto")
    texto = re.sub(r'¿\s+', '¿', texto)
    # Quita espacios antes de '?', '.' o ',' (ej: "texto ?" -> "texto?")
    texto = re.sub(r'\s+([?.,])', r'\1', texto)
    return texto

def tiene_pregunta_valida(texto):
    return "¿" in texto

def longitud_valida(texto):
    n_words = len(texto.split())
    return MIN_WORDS <= n_words <= MAX_WORDS

def es_oracion_limpia(texto):
    texto = texto.strip()
    if not texto:
        return False

    if any(c in SIGNOS_PROHIBIDOS for c in texto):
        return False

    if ".." in texto:
        return False

    if not (texto.endswith(".") or texto.endswith("?")):
        return False

    if texto.count("¿") != texto.count("?"):
        return False

    balance = 0
    for char in texto:
        if char == "¿":
            balance += 1
        elif char == "?":
            balance -= 1
        if balance < 0 or balance > 1:
            return False

    n_letras = sum(1 for c in texto if c.isalpha())
    if n_letras < len(texto) * 0.5:
        return False
        
    return True

In [36]:
rng = np.random.default_rng(SEED)

europarl_dataset = load_dataset(
    "Helsinki-NLP/europarl",
    "en-es",
    split="train",
    streaming=True,
)

N_PREG_EURO = 2000
N_NO_PREG_EURO = 10000

europarl_preguntas = []
europarl_no_preguntas = []

buffer_contexto = []
k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])

for ejemplo in europarl_dataset:
    texto = normalizar_espacios_puntuacion(ejemplo["translation"]["es"].strip())

    # Si una oración no es válida, se rompe la continuidad del discurso:
    # vaciamos el buffer actual si tenía algo y reiniciamos.
    if not longitud_valida(texto) or not es_oracion_limpia(texto):
        if buffer_contexto:
            bloque = " ".join(buffer_contexto)
            if tiene_pregunta_valida(bloque):
                if len(europarl_preguntas) < N_PREG_EURO:
                    europarl_preguntas.append(bloque)
            else:
                if len(europarl_no_preguntas) < N_NO_PREG_EURO:
                    europarl_no_preguntas.append(bloque)
            buffer_contexto = []
            k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])
        continue

    # Si es válida, la sumamos al párrafo en construcción (mismo discurso)
    buffer_contexto.append(texto)

    # Cuando alcanzamos las k oraciones consecutivas, cerramos el párrafo
    if len(buffer_contexto) == k_objetivo:
        bloque = " ".join(buffer_contexto)
        if tiene_pregunta_valida(bloque):
            if len(europarl_preguntas) < N_PREG_EURO:
                europarl_preguntas.append(bloque)
        else:
            if len(europarl_no_preguntas) < N_NO_PREG_EURO:
                europarl_no_preguntas.append(bloque)
        
        buffer_contexto = []
        k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])

    if len(europarl_preguntas) >= N_PREG_EURO and len(europarl_no_preguntas) >= N_NO_PREG_EURO:
        break

print(f"Europarl (en contexto) listo: {len(europarl_preguntas)} bloques con pregunta y {len(europarl_no_preguntas)} sin pregunta")

Europarl (en contexto) listo: 2000 bloques con pregunta y 10000 sin pregunta


In [37]:
opus100_dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-es",
    split="train",
    streaming=True,
)

OPUS100_PREG_TARGET = 2000
OPUS100_NO_PREG_TARGET = 10000

opus100_preguntas = []
opus100_no_preguntas = []

buffer_contexto = []
k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])

for ejemplo in opus100_dataset:
    texto = normalizar_espacios_puntuacion(ejemplo["translation"]["es"].strip())

    if not longitud_valida(texto) or not es_oracion_limpia(texto):
        if buffer_contexto:
            bloque = " ".join(buffer_contexto)
            if tiene_pregunta_valida(bloque):
                if len(opus100_preguntas) < OPUS100_PREG_TARGET:
                    opus100_preguntas.append(bloque)
            else:
                if len(opus100_no_preguntas) < OPUS100_NO_PREG_TARGET:
                    opus100_no_preguntas.append(bloque)
            buffer_contexto = []
            k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])
        continue

    buffer_contexto.append(texto)

    if len(buffer_contexto) == k_objetivo:
        bloque = " ".join(buffer_contexto)
        if tiene_pregunta_valida(bloque):
            if len(opus100_preguntas) < OPUS100_PREG_TARGET:
                opus100_preguntas.append(bloque)
        else:
            if len(opus100_no_preguntas) < OPUS100_NO_PREG_TARGET:
                opus100_no_preguntas.append(bloque)
                
        buffer_contexto = []
        k_objetivo = rng.choice([1, 2, 3, 4], p=[0.20, 0.40, 0.30, 0.10])

    if (len(opus100_preguntas) >= OPUS100_PREG_TARGET and
        len(opus100_no_preguntas) >= OPUS100_NO_PREG_TARGET):
        break

print(f"OPUS-100 listo: {len(opus100_preguntas)} bloques con pregunta y {len(opus100_no_preguntas)} sin pregunta")

OPUS-100 listo: 2000 bloques con pregunta y 10000 sin pregunta


### Combinando Datasets y concatenación sintetica

In [38]:
# Combinar datasets de manera random (ahora cada elemento ya es un bloque en contexto)
rng = np.random.default_rng(SEED)

N_PREG = int(N_TOTAL * 0.2)
N_NO_PREG = int(N_TOTAL - N_PREG)

# Preguntas: mitad de cada fuente
n_preg_opus100 = min(len(opus100_preguntas), N_PREG // 2)
n_preg_europarl = N_PREG - n_preg_opus100

rng.shuffle(europarl_preguntas)
rng.shuffle(opus100_preguntas)

preguntas_final = (
    opus100_preguntas[:n_preg_opus100] +
    europarl_preguntas[:n_preg_europarl]
)

# No-preguntas: mitad de cada fuente 
n_no_preg_opus100 = min(len(opus100_no_preguntas), N_NO_PREG // 2)
n_no_preg_europarl = N_NO_PREG - n_no_preg_opus100

rng.shuffle(europarl_no_preguntas)
rng.shuffle(opus100_no_preguntas)

no_preguntas_final = (
    opus100_no_preguntas[:n_no_preg_opus100] +
    europarl_no_preguntas[:n_no_preg_europarl]
)

# Creación del DataFrame final 
df_preguntas = pd.DataFrame({
    "Text": preguntas_final,
    "tiene_pregunta": 1,
    "fuente": (
        ["opus100"] * n_preg_opus100 + 
        ["europarl"] * n_preg_europarl
    )
})

df_no_preguntas = pd.DataFrame({
    "Text": no_preguntas_final,
    "tiene_pregunta": 0,
    "fuente": (
        ["opus100"] * n_no_preg_opus100 + 
        ["europarl"] * n_no_preg_europarl
    )
})

df = pd.concat(
    [df_preguntas, df_no_preguntas], ignore_index=True
).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Total de instancias en df: {len(df)}")
print(df.groupby(["fuente", "tiene_pregunta"]).size())

Total de instancias en df: 15000
fuente    tiene_pregunta
europarl  0                 6000
          1                 1500
opus100   0                 6000
          1                 1500
dtype: int64


Dado que *Europarl* y *OPUS-100* están pre-segmentados a nivel de oración individual, entrenar directamente con estas filas introduciría un **sesgo posicional**. Las redes podrían aprender erróneamente que los signos de cierre (`.` y `?`) ocurren casi exclusivamente en el último token de la secuencia.

Se podría concatenar oraciones al azar, una vez mezclados los datasets, para lograr evitar el sesgo posicional. Sin embargo, concatenar de esta forma generaría saltos temáticos artificiales en cada límite de oración, corriendo el riesgo de que las redes recurrentes asocien el punto seguido exclusivamente con un cambio brusco de contexto semántico y fallen al puntuar párrafos cohesivos.


Para resolver ambos problemas, se implementó una agrupación mediante *buffers* de **1 a 4 oraciones consecutivas válidas** directamente durante la lectura secuencial del *stream*:
1. **Continuidad semántica y de registro:** En *Europarl*, unir oraciones contiguas reconstruye fragmentos reales de un mismo discurso parlamentario, manteniendo el hilo conductor. Si una oración intermedia es descartada por los filtros de limpieza, el *buffer* se cierra inmediatamente para evitar unir oraciones con texto faltante entre sí.
2. **Aislamiento de dominios:** Nunca se cruzan oraciones de *Europarl* con *OPUS-100* dentro de una misma instancia. De este modo, el modelo aprende a detectar límites de oración (puntos seguidos y `?` internos) guiándose por la estructura sintáctica natural dentro de un mismo contexto y estilo léxico.

DETALLE: opus100 no tiene continuidad y hay cambios bruscos de contexto potencialmente, quiza conviene quedarnos con europarl o que sea una mayoría en el entrenamiento.


### Chequeo de clases

In [39]:
textos = df['Text'].dropna().astype(str).str.strip()
total_oraciones = len(textos)

contiene_apertura = textos.str.contains(r'¿', regex=False)
contiene_cierre = textos.str.contains(r'\?', regex=True)
contiene_coma = textos.str.contains(r',', regex=False)


contiene_punto_final = textos.str.contains(r'\.$', regex=True)

contiene_punto_seguido = textos.str.contains(r'\.(?!$)', regex=True)

sin_puntuacion = ~(contiene_apertura | contiene_cierre | contiene_coma | contiene_punto_final | contiene_punto_seguido)

apariciones = {
    '¿ (Apertura de pregunta)': textos.str.count(r'¿').sum(),
    '? (Cierre de pregunta)': textos.str.count(r'\?').sum(),
    ', (Coma)': textos.str.count(r',').sum(),
    '. (Punto final)': textos.str.count(r'\.$').sum(),
    '. (Punto seguido)': textos.str.count(r'\.(?!$)').sum(),
    '0 (Sin puntuación)': sin_puntuacion.sum()
}

resultados = {
    'Clase': [
        '¿ (Apertura de pregunta)', 
        '? (Cierre de pregunta)', 
        ', (Coma)', 
        '. (Punto final)', 
        '. (Punto seguido)', 
        '0 (Sin puntuación)'
    ],
    'Oraciones con la clase': [
        contiene_apertura.sum(),
        contiene_cierre.sum(),
        contiene_coma.sum(),
        contiene_punto_final.sum(),
        contiene_punto_seguido.sum(),
        sin_puntuacion.sum()
    ]
}

df_res = pd.DataFrame(resultados)

df_res['% de oraciones'] = (df_res['Oraciones con la clase'] / total_oraciones * 100).round(2)
df_res['Apariciones totales'] = df_res['Clase'].map(apariciones)

df_res = df_res.sort_values(by='% de oraciones', ascending=False)

print(f"Total de oraciones analizadas: {total_oraciones}\n")
print(df_res.to_string(index=False))

Total de oraciones analizadas: 15000

                   Clase  Oraciones con la clase  % de oraciones  Apariciones totales
         . (Punto final)                   12855           85.70                12855
                , (Coma)                    9920           66.13                23505
       . (Punto seguido)                    6342           42.28                 9289
¿ (Apertura de pregunta)                    3000           20.00                 3338
  ? (Cierre de pregunta)                    3000           20.00                 3338
      0 (Sin puntuación)                       0            0.00                    0


### Ejemplos del Dataset

In [40]:
# ─ Muestra del dataset ──────────────────────────────────────────────────────
print("Ejemplos de PREGUNTAS (OPUS-100):")
for t in df[(df["tiene_pregunta"]==1) & (df["fuente"]=="opus100")]["Text"].head(3):
    print(f"  → {t}")

print("\nEjemplos de PREGUNTAS (Europarl):")
for t in df[(df["tiene_pregunta"]==1) & (df["fuente"]=="europarl")]["Text"].head(3):
    print(f"  → {t}")

print("\nEjemplos de NO-PREGUNTAS (OPUS-100):")
for t in df[(df["tiene_pregunta"]==0) & (df["fuente"]=="opus100")]["Text"].head(3):
    print(f"  → {t}")

print("\nEjemplos de NO-PREGUNTAS (Europarl):")
for t in df[(df["tiene_pregunta"]==0) & (df["fuente"]=="europarl")]["Text"].head(3):
    print(f"  → {t}")

Ejemplos de PREGUNTAS (OPUS-100):
  → Espera, ¿por qué está tu camiseta en la cama de Julia? Los teléfonos móviles y ver actores porno con grandes penes.
  → ¿Y puede confirmar que ambos estuvieron allí toda la noche?
  → Para ello es necesario que la Comisión no llene sólo los cajones con sus últimas propuestas sobre empleo, medio ambiente e iniciativas locales de empleo, sino que exija realmente su puesta en práctica como uso eficiente del dinero europeo. ¿Qué me dices de todo el dinero que ganan sus monjes?

Ejemplos de PREGUNTAS (Europarl):
  → ¿Cuál es ahora la situación, al cabo de ocho meses y después de la reunión de Biarritz? Más allá de la esperanza de lograr algún progreso en relación con el tema de la ampliación de la mayoría cualificada, en realidad hemos avanzado muy poco.
  → Sin embargo esta sociedad libre de drogas jamás existirá. ¿O también ha de prohibirse el consumo de cigarrillos y alcohol?
  → Así nos lo enseñan los hechos del caso Leonardo, en el que los interven

# Construcción de datos para entrenar modelos

In [51]:
VOCALES = set("aeiouáéíóúü")
VOCALES_SIN_TILDE = set("aeiouü")

ADVERBIOS_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cómo", "cuándo", "dónde",
    "cuánto", "cuánta", "cuántos", "cuántas", "cuál", "cuáles",
}

PRONOMBRES_INTERROGATIVOS = {
    "qué", "quién", "quiénes", "cuál", "cuáles", "cuánto",
    "cuánta", "cuántos", "cuántas",
}

# Unimos ambos para el atributo "es_palabra_interrogativa"
PALABRAS_INTERROGATIVAS = ADVERBIOS_INTERROGATIVOS | PRONOMBRES_INTERROGATIVOS

def _es_palabra_interrogativa(palabra: str) -> int:
    return int(palabra in PALABRAS_INTERROGATIVAS)

def separar_oraciones(texto: str) -> list[str]:
    """Separa un texto en oraciones usando . y ? como fin de oración."""
    partes = [o.strip() for o in re.split(r"(?<=[.?])\s+", texto.strip()) if o.strip()]
    return partes if partes else [texto.strip()]

In [52]:
# Función de etiquetado
def construir_dataset_etiquetado(textos):
    """
    Recibe una secuencia de instancias (por ejemplo, df['Text']) y devuelve
    un DataFrame a nivel palabra con IDs jerárquicos:
    id_texto -> id_oracion -> id_palabra.
    """
    filas = []

    for id_texto, texto in enumerate(textos):
        if pd.isna(texto):
            continue

        texto = str(texto).strip()
        if not texto:
            continue

        oraciones = separar_oraciones(texto)

        for id_oracion, oracion in enumerate(oraciones):
            id_palabra = 0
            for palabra_cruda in oracion.split():
                palabra_sin_signos = re.sub(r"[¿?.,]", "", palabra_cruda)
                if not palabra_sin_signos:
                    continue

                empieza_con_mayuscula = int(palabra_sin_signos[0].isupper())
                palabra_limpia = palabra_sin_signos.lower()

                filas.append({
                    "id_texto": id_texto,
                    "id_oracion": id_oracion,
                    "id_palabra": id_palabra,
                    "palabra": palabra_cruda,
                    "palabra_limpia": palabra_limpia,
                    "tiene_¿": int("¿" in palabra_cruda),
                    "tiene_?": int("?" in palabra_cruda),
                    "tiene_.": int("." in palabra_cruda),
                    "tiene_,": int("," in palabra_cruda),
                    "empieza_con_mayuscula": empieza_con_mayuscula,
                })
                id_palabra += 1

    return pd.DataFrame(filas)

In [54]:
# Función de atributos
def _tiene_tilde(palabra: str) -> int:
    """1 si la palabra contiene al menos una vocal con tilde (á é í ó ú)."""
    return int(any(ch in "áéíóú" for ch in palabra))


def _cantidad_vocales(palabra: str) -> int:
    return sum(1 for ch in palabra if ch in VOCALES)


def _cantidad_consonantes(palabra: str) -> int:
    """Cuenta letras que no son vocales. Ignora dígitos, espacios, etc."""
    return sum(
        1 for ch in palabra
        if unicodedata.category(ch).startswith("L") and ch not in VOCALES
    )


def añadir_atributos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega atributos léxicos y contextuales a un DataFrame producido por
    construir_dataset_etiquetado.
    """
    df = df.copy()

    columnas_requeridas = [
        "id_texto", "id_oracion", "id_palabra", "palabra_limpia", "tiene_¿", "tiene_?",
    ]
    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en el DataFrame: {faltantes}")

    # Orden explícito para que los shifts prev/next sean correctos.
    df = df.sort_values(["id_texto", "id_oracion", "id_palabra"]).reset_index(drop=True)

    # Atributos por palabra
    df["distancia_inicio"] = df["id_palabra"]
    df["tiene_tilde"] = df["palabra_limpia"].map(_tiene_tilde)
    df["cantidad_vocales"] = df["palabra_limpia"].map(_cantidad_vocales)
    df["cantidad_consonantes"] = df["palabra_limpia"].map(_cantidad_consonantes)
    df["cantidad_letras"] = df["cantidad_vocales"] + df["cantidad_consonantes"]

    df["es_adverbio_interrogativo"] = df["palabra_limpia"].map(_es_palabra_interrogativa)
    df["es_palabra_interrogativa"] = df["es_adverbio_interrogativo"]

    # Contexto (palabra anterior/siguiente por oración dentro de cada texto).
    group_cols = ["id_texto", "id_oracion"]
    for col in ["es_palabra_interrogativa", "tiene_tilde", "tiene_¿", "tiene_?"]:
        df[f"prev_{col}"] = df.groupby(group_cols)[col].shift(1).fillna(0).astype(int)
        df[f"next_{col}"] = df.groupby(group_cols)[col].shift(-1).fillna(0).astype(int)

    return df

In [55]:
# Etiquetar primeras 10 instancias de df y guardar en df_etiquetado
textos_10 = df["Text"].dropna().astype(str).head(10).tolist()

df_etiquetado = construir_dataset_etiquetado(textos_10)
df_etiquetado = añadir_atributos(df_etiquetado)

n_oraciones = df_etiquetado[["id_texto", "id_oracion"]].drop_duplicates().shape[0]
print(f"Instancias usadas (id_texto): {len(textos_10)}")
print(f"Oraciones detectadas: {n_oraciones}")
print(f"Filas etiquetadas: {len(df_etiquetado)}")
df_etiquetado.head(20)

Instancias usadas (id_texto): 10
Oraciones detectadas: 16
Filas etiquetadas: 209


,id_texto,id_oracion,id_palabra,palabra,palabra_limpia,tiene_¿,tiene_?,tiene_.,"tiene_,",empieza_con_mayuscula,...,es_adverbio_interrogativo,es_palabra_interrogativa,prev_es_palabra_interrogativa,next_es_palabra_interrogativa,prev_tiene_tilde,next_tiene_tilde,prev_tiene_¿,next_tiene_¿,prev_tiene_?,next_tiene_?
0,0,0,0,sabe,sabe,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,que,que,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,2,yo,yo,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,3,creo,creo,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,4,que,que,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
5,0,0,5,llegó,llegó,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,6,hasta,hasta,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0
7,0,0,7,aquí.,aquí,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
8,1,0,0,¿Cuál,cuál,1,0,0,0,1,...,1,1,0,0,0,0,0,0,0,0
9,1,0,1,es,es,0,0,0,0,0,...,0,0,1,0,1,0,1,0,0,0


# Old

In [ ]:
X_train, y_train, X_control, y_control = train_test_split(df_feats2, 'label')
df_control = pd.concat([X_control, y_control], axis=1)
df_dev = pd.concat([X_train, y_train], axis=1)

In [ ]:
def es_signo_puntuacion(token: str):
    # True si el token es (o contiene solo) signos de puntuación
    if not token or token.startswith("##"):
        return False
    return all(unicodedata.category(ch).startswith("P") for ch in token)


def construir_dataset_interrogacion(
    oraciones,
    tokenizer,
    *,
    quitar_toda_puntuacion: bool = True,
    max_tokens: int = 512,
    ventana: int | None = None,
):
    """
    Construye una lista de instancias token-a-token para predecir '¿'
    a partir de texto normalizado.

    Cada instancia es un dict:
        {
            'oracion_id': int,
            'token': str,
            'token_id': int,
            'distancia_inicio': int,   # cuántos tokens no-'¿' hay antes
            'label': int,              # 1 si estaba precedido por '¿'
            'ventana': list[int|None], # solo si ventana no es None
        }
    """
    registros = []

    for oracion_id, oracion in enumerate(oraciones):
        # 1) Tokenizar la oración original (con su puntuación)
        tokens = tokenizer.tokenize(oracion)
        if not tokens:
            continue

        # 2) Truncar al límite de BERT
        tokens = tokens[:max_tokens]

        # 3) Construir input limpio + etiquetas
        tokens_limpios = []
        target = []
        for i, tok in enumerate(tokens):
            if tok == "¿":
                continue  # '¿' es el target: nunca va al input
            # opcional: quitamos también ? . , etc. Si
            # si encadenamos los clasificadores creo que podría servir ¿?
            if quitar_toda_puntuacion and es_signo_puntuacion(tok):
                continue  
            tokens_limpios.append(tok)
            # etiqueta: ¿el token anterior (en el original) era '¿'?
            target.append(1 if (i > 0 and tokens[i - 1] == "¿") else 0)

        # Por si no habían tokens ¿?
        if not tokens_limpios:
            continue

        # 4) Convertir a ids (BERT ya tiene embeddings: solo pasamos input_ids)
        ids = tokenizer.convert_tokens_to_ids(tokens_limpios)

        # 5) Armar registros (con o sin ventana de contexto)
        n = len(ids)
        for i in range(n):
            registro = {
                "oracion_id": oracion_id, # esto no lo había pensado pero se ve bien...
                "token": tokens_limpios[i],
                "token_id": ids[i],
                "distancia_inicio": i,
                "label": target[i],
            }
            if ventana is not None:
                vecinos = [
                    ids[j] if 0 <= j < n else None
                    for j in range(i - ventana, i + ventana + 1)
                ]
                registro["ventana"] = vecinos
            registros.append(registro)

    return registros

In [ ]:
def es_signo_puntuacion(palabra: str):
    if not palabra or palabra.startswith("##"):
        return False
    return all(unicodedata.category(ch).startswith("P") for ch in palabra)


def normalizar_texto(texto: str) -> list[str]:
    """Devuelve la lista de palabras normalizadas (minúsculas, sin puntuación)."""
    # 1) Separar por espacios en blanco (cubre espacios, tabs, saltos de línea)
    palabras = texto.split()

    resultado = []
    for palabra in palabras:
        # 2) Quitar signos de puntuación que estén pegados a la palabra
        #    (al principio, al final o en el medio). No quitamos letras,
        #    números, ni marcas diacríticas (á, ñ, ü...).
        limpia = "".join(
            ch for ch in palabra
            if not unicodedata.category(ch).startswith("P")
        )
        # 3) Descartar si quedó vacía (era solo puntuación, ej. '¿', '?', '...')
        if not limpia:
            continue
        # 4) Minúsculas. OJO: .lower() respeta acentos en Python 3.
        resultado.append(limpia.lower())

    return resultado


def normalizar(texto: str) -> str:
    """Versión string del normalizador: '¿Hola? Tonto.' -> 'hola tonto'."""
    return " ".join(normalizar_texto(texto))

#def construir_dataset_interrogacion(
#    oraciones: list[str],
#    tokenizer,
#    *,
#    quitar_toda_puntuacion: bool = True,
#    max_tokens: int = 512,
#    ventana: int | None = None,
#):
#    """
#    Construye una lista de instancias token-a-token para predecir '¿'
#    a partir de texto normalizado.
#
#    Cada instancia es un dict:
#        {
#            'oracion_id': int,
#            'token': str,
#            'token_id': int,
#            'distancia_inicio': int,   # cuántos tokens no-'¿' hay antes
#            'label': int,              # 1 si estaba precedido por '¿'
#            'ventana': list[int|None], # solo si ventana no es None
#        }
#    """
#    registros = []
#
#    for oracion_id, oracion in enumerate(oraciones):
#        # 1) Tokenizar la oración original (con su puntuación)
#        # 3) Construir input limpio + etiquetas
#        palabras_limpias = []
#        target = []
#        palabras = oracion.split(" ")
#        for i, p in enumerate(palabras):
#            if p in "¿":
#                continue  # '¿' es el target: nunca va al input
#            # opcional: quitamos también ? . , etc. Si
#            # si encadenamos los clasificadores creo que podría servir ¿?
#            if quitar_toda_puntuacion and es_signo_puntuacion(p):
#                continue  
#            palabras_limpias.append(p[1:])
#            # etiqueta: ¿el token anterior (en el original) era '¿'?
#            target.append(1 if (i > 0 and tokens[i - 1] == "¿") else 0)
#
#
#        n = len(palabras)
#        for i in range(n):
#            registro = {
#                "oracion_id": oracion_id, # esto no lo había pensado pero se ve bien...
#                "token": palabras_limpias[i],
#                "token_id": ids[i],
#                "distancia_inicio": i,
#                "label": target[i],
#            }
#            if ventana is not None:
#                vecinos = [
#                    ids[j] if 0 <= j < n else None
#                    for j in range(i - ventana, i + ventana + 1)
#                ]
#                registro["ventana"] = vecinos
#            registros.append(registro)
#
#    return registros

In [ ]:
normalizar("¿Tomi vino gracioso del parcial?, Si.")

In [ ]:
# A) Con ventana de contexto (para modelos no contextuales / baseline)
# se podría usar la lista en sí y no crear df_balanceado

registros_ventana_5 = construir_dataset_interrogacion(
    df_balanceado["Text"].tolist(),
    tokenizer,
    quitar_toda_puntuacion=True,
    ventana=5
)

# C) Convertir a DataFrame para inspeccionar
df_feats = pd.DataFrame(registros_ventana_5)
print(df_feats.head())
print(df_feats["label"].value_counts(normalize=True))  # desbalance esperado

In [ ]:
"¿juancito fue a comer.".split(" ")
x = "abc"
x[0]

In [ ]:
# probar con o sin oracion_id
df_feats2 = df_feats.drop(columns=['token'])

df_feats2

In [ ]:
df_feats2['label']

In [ ]:
print(df_feats2.describe())

In [24]:
from sklearn.model_selection import StratifiedKFold


SEED = 2026
np.random.seed(SEED)

# Definimos el K-Fold estratificado para usar en la validación cruzada de los modelos
stratifiedKFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [25]:
# Función para separar el dataset en train y test, manteniendo la proporción de clases
def train_test_split(df, target):
    buen_pronostico = df[df[target] == 1]
    p = buen_pronostico.shape[0]/df.shape[0]

    mal_pronostico = df[df[target] == 0]
    buenos = len(buen_pronostico)/len(df)
    filasBuenas = int(len(df)*buenos*p)
    filasMalas = int(len(df)*p-filasBuenas)
    # buscar como selecciona .sample()
    test = pd.DataFrame(columns=buen_pronostico.columns).astype(buen_pronostico.dtypes)
    test = pd.concat([test, buen_pronostico.sample(n=filasBuenas, random_state=SEED)], ignore_index=False)
    test = pd.concat([test, mal_pronostico.sample(n=filasMalas, random_state=SEED)], ignore_index=False)

    train = df.drop(test.index, axis=0)    
    return train.drop(columns=[target]), train[target], test.drop(columns=[target]), test[target]

In [26]:
X_train, y_train, X_control, y_control = train_test_split(df_feats2, 'label')
df_control = pd.concat([X_control, y_control], axis=1)
df_dev = pd.concat([X_train, y_train], axis=1)

In [27]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC


def mejor_SVC_dado_df(X_train, y_train):
    param_dist = [
        {
            'estimator__kernel': ['rbf'],
            'estimator__C': np.logspace(-3, 3, 7), 
            'estimator__gamma': list(np.logspace(-4, 1, 6)) + ['scale', 'auto'], 
        },
        {
            'estimator__kernel': ['poly'],
            'estimator__C': np.logspace(-3, 3, 7),
            'estimator__degree': [2, 3, 5],
            'estimator__coef0': [0.0, 0.1, 0.5],
        },
        {
            'estimator__kernel': ['linear'],
            'estimator__C': np.logspace(-3, 3, 7),
        },
    ]

    modelo_base = CalibratedClassifierCV(SVC(), ensemble=False)

    random_search_svc = RandomizedSearchCV(
        modelo_base,
        param_distributions=param_dist,
        n_iter=30,
        scoring='roc_auc',
        cv=stratifiedKFold,
        random_state=SEED,
        n_jobs=-1
    )
    
    random_search_svc.fit(X_train, y_train)
    return random_search_svc

In [ ]:
random_search_svc = mejor_SVC_dado_df(X_train, y_train)
random_search_svc.best_params_

In [ ]:
# Resultados del modelo SVC con su mejor combinación de hiperparámetros más otras métricas de evaluación
resultado_svc = stratifiedKFoldCV(random_search_svc.best_estimator_, X_train, y_train, stratifiedKFold)

resultado_svc.informe

# Consulta
- Sobre añadir datos sinteticos para tener más preguntas ('¿' '?').
- Sobre concatenar oraciones. (Para que el modelo tenga que separarlas) (En parte, hay que dejar oraciones únicas).
  
- Sobre parseo de oraciones y balanceo de clases
- sobre multiclases, que nos conviene?

# TODO
- chequear balanceo de clases
- realizar dt y svm o knn
- falta atributo de signo de pregunta en el texto